In [1]:
import pandas as pd
from google.cloud import bigquery

BQ_PROJECT = "cred-1556636033881"
BQ_TABLE_OUT = "temporary.BrandRankerFilteredCompanyIds"

full_df = pd.read_csv("BRAND RANKER FINAL-20260219-161826-Internal - results-20260219-161826.csv")
exclude_df = pd.read_csv("Full UTA - Data QA Analysis - suggestions-to-remove.csv")

print(f"Full list: {len(full_df)} companies")
print(f"Exclusion list: {len(exclude_df)} companies")

exclude_ids = set(exclude_df["companyId"].astype(int))
filtered_df = full_df[~full_df["CompanyID"].astype(int).isin(exclude_ids)].copy()
filtered_df = filtered_df.rename(columns={"CompanyID": "companyId", "CompanyName": "companyName", "IndustryName": "industryName"})

print(f"After exclusion: {len(filtered_df)} companies ({len(full_df) - len(filtered_df)} removed)")
filtered_df.head()

Full list: 9952 companies
Exclusion list: 1683 companies
After exclusion: 8269 companies (1683 removed)


,companyId,companyName,industryName
1,224881,Lidl US,Supermarkets & Grocery Stores
2,219521,Browning,Leisure Products
3,224043,Condat,Basic Materials Manufacturing
4,227472,Noritz America,Wholesale
5,223503,"Cacique, Inc.",Food Manufacturing & Processing


In [2]:
client = bigquery.Client(project=BQ_PROJECT)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
)

job = client.load_table_from_dataframe(filtered_df, BQ_TABLE_OUT, job_config=job_config)
job.result()

table = client.get_table(BQ_TABLE_OUT)
print(f"Uploaded {table.num_rows} rows to {BQ_TABLE_OUT}")

Uploaded 8269 rows to temporary.BrandRankerFilteredCompanyIds


In [3]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import spearmanr

pio.renderers.default = "notebook_connected"

def bq_to_df(client, sql):
    return client.query(sql).to_dataframe()

df_2026 = bq_to_df(client, """
    SELECT A.*
    FROM data_science_results_company.MarketingBreakdownTopDownAllocationModel A
    JOIN temporary.BrandRankerFilteredCompanyIds B
    ON A.companyId = B.companyId
""")

df_2025 = bq_to_df(client, """
    SELECT A.*
    FROM data_science_results_company.MarketingBreakdownTopDownAllocationModel20250101 A
    JOIN temporary.BrandRankerFilteredCompanyIds B
    ON A.companyId = B.companyId
""")

print(f"2026 rows: {len(df_2026)}")
print(f"2025 rows: {len(df_2025)}")

/Users/sandeepgiri/miniconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


2026 rows: 8256
2025 rows: 8257


In [4]:
PREDICTION_COLS = [
    'PredictedMarketingBudget', 'PredictedSalesExpense',
    'PredictedMarketingExpense', 'PredictedInfluencerSpend',
    'PredictedSponsorshipSpend', 'PredictedAdvertisingSpend',
    'PredictedResidualSpend',
]

merged = df_2025.merge(df_2026, on='companyId', suffixes=['_2025', '_2026'])
print(f"Matched companies across both years: {len(merged)}\n")

print("=== Median Year-over-Year Ratio (2026 / 2025) ===")
for col in PREDICTION_COLS:
    ratio = merged[f"{col}_2026"] / merged[f"{col}_2025"]
    print(f"  {col}: {np.nanmedian(ratio):.4f}")

print("\n=== Spearman Rank Correlation (2025 vs 2026) ===")
for col in PREDICTION_COLS:
    rho, pval = spearmanr(
        merged[f"{col}_2025"],
        merged[f"{col}_2026"],
        nan_policy="omit"
    )
    print(f"  {col}: rho={rho:.4f}, p={pval:.2e}")

Matched companies across both years: 8256

=== Median Year-over-Year Ratio (2026 / 2025) ===
  PredictedMarketingBudget: 0.9979
  PredictedSalesExpense: 0.9979
  PredictedMarketingExpense: 0.9979
  PredictedInfluencerSpend: 1.0000
  PredictedSponsorshipSpend: 1.0000
  PredictedAdvertisingSpend: 0.9975
  PredictedResidualSpend: 0.9979

=== Spearman Rank Correlation (2025 vs 2026) ===
  PredictedMarketingBudget: rho=0.8182, p=0.00e+00
  PredictedSalesExpense: rho=0.8541, p=0.00e+00
  PredictedMarketingExpense: rho=0.8402, p=0.00e+00
  PredictedInfluencerSpend: rho=0.8657, p=0.00e+00
  PredictedSponsorshipSpend: rho=0.8352, p=0.00e+00
  PredictedAdvertisingSpend: rho=0.8310, p=0.00e+00
  PredictedResidualSpend: rho=0.8479, p=0.00e+00


In [6]:
for col in PREDICTION_COLS:
    x_2025 = df_2025[col].replace([np.inf, -np.inf], np.nan).dropna()
    x_2026 = df_2026[col].replace([np.inf, -np.inf], np.nan).dropna()

    fig = go.Figure()
    fig.add_trace(go.Histogram(x=x_2025, name="2025", opacity=0.6))
    fig.add_trace(go.Histogram(x=x_2026, name="2026", opacity=0.6))
    fig.update_layout(
        title=f"Distribution Comparison: {col}",
        barmode='overlay',
        xaxis_title=col,
        yaxis_title="Count",
        template="plotly_white",
    )
    fig.show()

In [7]:
for col in PREDICTION_COLS:
    x_2025 = np.log1p(df_2025[col].replace([np.inf, -np.inf], np.nan).dropna())
    x_2026 = np.log1p(df_2026[col].replace([np.inf, -np.inf], np.nan).dropna())

    fig = go.Figure()
    fig.add_trace(go.Histogram(x=x_2025, name="2025", opacity=0.6))
    fig.add_trace(go.Histogram(x=x_2026, name="2026", opacity=0.6))
    fig.update_layout(
        title=f"Log-Scale Distribution: {col}",
        barmode='overlay',
        xaxis_title=f"log1p({col})",
        yaxis_title="Count",
        template="plotly_white",
    )
    fig.show()

In [8]:
name_lookup = filtered_df[['companyId', 'companyName', 'industryName']].drop_duplicates()

value_cols = [c for c in PREDICTION_COLS if c in df_2026.columns]
meta_cols = ['companyId']

df_2026_named = df_2026[meta_cols + value_cols].merge(name_lookup, on='companyId', how='left')
df_2025_named = df_2025[meta_cols + value_cols].merge(name_lookup, on='companyId', how='left')

top_n = 30
print(f"=== Top {top_n} Companies by PredictedMarketingBudget (2026) ===\n")
top_2026 = df_2026_named.nlargest(top_n, 'PredictedMarketingBudget')
display_cols = ['companyName', 'industryName'] + value_cols
top_2026[display_cols].style.format({c: '${:,.0f}' for c in value_cols})

=== Top 30 Companies by PredictedMarketingBudget (2026) ===



,companyName,industryName,PredictedMarketingBudget,PredictedSalesExpense,PredictedMarketingExpense,PredictedInfluencerSpend,PredictedSponsorshipSpend,PredictedAdvertisingSpend,PredictedResidualSpend
8162,Canon Inc.,Office Supplies & Equipment,"$140,887,020,856","$93,924,680,571","$136,783,515,394","$10,884,778,324","$20,491,283,562","$30,176,520,042","$75,230,933,467"
3592,Verizon,Telecommunications,"$126,946,161,696","$155,156,419,851","$123,248,700,676","$3,642,452,991","$18,827,685,694","$26,829,341,585","$73,949,220,406"
1597,Amazon,Online Retailers,"$93,168,125,777","$62,112,083,852","$90,454,491,046","$2,417,745,270","$8,021,028,433","$30,265,747,267","$49,749,970,075"
1588,General Motors,Automotive,"$92,173,402,612","$75,414,602,137","$94,054,492,462","$3,369,270,435","$15,300,962,846","$18,951,563,704","$56,432,695,477"
4964,Alphabet,IT Services,"$71,957,120,454","$133,634,652,271","$70,546,196,523","$1,011,716,892","$9,683,170,047","$10,468,972,018","$49,382,337,566"
4894,GSK,Pharmaceuticals,"$46,272,031,710","$118,985,224,396","$47,216,358,887","$571,349,007","$3,679,076,162","$7,553,664,553","$35,412,269,165"
2331,Christian Dior Couture,Luxury Goods,"$44,503,932,807","$36,412,308,660","$45,412,176,333","$3,029,255,128","$6,157,189,650","$8,978,425,755","$27,247,305,800"
2153,Intel Corporation,Chips & Semiconductors,"$33,089,281,767","$61,451,523,281","$32,440,472,320","$713,907,426","$4,033,543,438","$4,984,690,831","$22,708,330,624"
3539,ABB E-mobility,Specialty Manufacturing,"$32,537,994,349","$130,151,977,395","$35,367,385,162","$840,729,842","$2,758,495,971","$3,474,251,220","$28,293,908,129"
4234,Netflix,Streaming services,"$28,768,079,599","$35,160,986,176","$27,930,174,368","$1,861,772,813","$2,400,605,707","$6,909,691,227","$16,758,104,621"


In [9]:
comparison = df_2025_named[meta_cols + value_cols + ['companyName', 'industryName']].merge(
    df_2026_named[meta_cols + value_cols],
    on='companyId',
    suffixes=['_2025', '_2026'],
)

for col in value_cols:
    comparison[f"{col}_pctChange"] = (
        (comparison[f"{col}_2026"] - comparison[f"{col}_2025"])
        / comparison[f"{col}_2025"].replace({0: np.nan})
        * 100
    )

print(f"=== Top {top_n} Companies: 2025 vs 2026 PredictedMarketingBudget ===\n")
top_compare = comparison.nlargest(top_n, 'PredictedMarketingBudget_2026')
show_cols = [
    'companyName', 'industryName',
    'PredictedMarketingBudget_2025', 'PredictedMarketingBudget_2026', 'PredictedMarketingBudget_pctChange',
    'PredictedMarketingExpense_2025', 'PredictedMarketingExpense_2026', 'PredictedMarketingExpense_pctChange',
]
top_compare[show_cols].style.format({
    c: '${:,.0f}' for c in show_cols if c.endswith(('_2025', '_2026'))
}).format({
    c: '{:+.1f}%' for c in show_cols if c.endswith('_pctChange')
})

=== Top 30 Companies: 2025 vs 2026 PredictedMarketingBudget ===



,companyName,industryName,PredictedMarketingBudget_2025,PredictedMarketingBudget_2026,PredictedMarketingBudget_pctChange,PredictedMarketingExpense_2025,PredictedMarketingExpense_2026,PredictedMarketingExpense_pctChange
728,Canon Inc.,Office Supplies & Equipment,124062443350.151947,140887020855.758575,+13.6%,120448974126.361115,136783515393.940369,+13.6%
4254,Verizon,Telecommunications,184287599999.999969,126946161696.486374,-31.1%,178919999999.999969,123248700676.200363,-31.1%
5127,Amazon,Online Retailers,2713452600000.000000,93168125777.308670,-96.6%,2634420000000.000000,90454491045.930740,-96.6%
4181,General Motors,Automotive,177870000000.000000,92173402612.379074,-48.2%,181500000000.000000,94054492461.611298,-48.2%
3417,Alphabet,IT Services,992745599999.999878,71957120453.825180,-92.8%,973279999999.999878,70546196523.358017,-92.8%
2098,GSK,Pharmaceuticals,69490968401.406342,46272031709.519859,-33.4%,70909151430.006470,47216358887.265167,-33.4%
2961,Christian Dior Couture,Luxury Goods,89924640605.276367,44503932806.683670,-50.5%,91759837352.322830,45412176333.350685,-50.5%
6853,Intel Corporation,Chips & Semiconductors,30559200000.000000,33089281766.531605,+8.3%,29960000000.000000,32440472320.129025,+8.3%
6893,ABB E-mobility,Specialty Manufacturing,32063259205.614422,32537994348.846909,+1.5%,34851368701.754807,35367385161.790115,+1.5%
826,Netflix,Streaming services,135228627899.999985,28768079598.935371,-78.7%,131289929999.999985,27930174367.898418,-78.7%


In [10]:
print("=== Summary Statistics by Year ===\n")
for col in value_cols:
    s25 = df_2025_named[col].dropna()
    s26 = df_2026_named[col].dropna()
    print(f"{col}:")
    print(f"  2025 — median: ${np.median(s25):,.0f}  |  mean: ${np.mean(s25):,.0f}  |  p25: ${np.percentile(s25, 25):,.0f}  |  p75: ${np.percentile(s25, 75):,.0f}")
    print(f"  2026 — median: ${np.median(s26):,.0f}  |  mean: ${np.mean(s26):,.0f}  |  p25: ${np.percentile(s26, 25):,.0f}  |  p75: ${np.percentile(s26, 75):,.0f}")
    print()

=== Summary Statistics by Year ===

PredictedMarketingBudget:
  2025 — median: $298,582,244  |  mean: $2,240,048,362  |  p25: $217,795,461  |  p75: $499,831,101
  2026 — median: $280,152,798  |  mean: $437,425,327  |  p25: $210,236,918  |  p75: $388,192,205

PredictedSalesExpense:
  2025 — median: $435,241,802  |  mean: $2,963,496,707  |  p25: $259,408,872  |  p75: $675,776,143
  2026 — median: $396,471,217  |  mean: $639,460,884  |  p25: $235,121,095  |  p75: $548,797,021

PredictedMarketingExpense:
  2025 — median: $272,519,235  |  mean: $2,215,783,475  |  p25: $196,275,527  |  p75: $469,065,759
  2026 — median: $253,847,004  |  mean: $417,151,511  |  p25: $191,381,012  |  p75: $364,069,008

PredictedInfluencerSpend:
  2025 — median: $7,817,513  |  mean: $55,748,986  |  p25: $3,758,713  |  p75: $16,030,089
  2026 — median: $7,406,096  |  mean: $16,057,615  |  p25: $3,439,978  |  p75: $14,662,015

PredictedSponsorshipSpend:
  2025 — median: $38,109,141  |  mean: $234,283,595  |  p25: 

In [17]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime as dt
import webbrowser, os

# ── Data prep with explicit float casting (BigQuery returns Decimal types) ──
name_lookup = filtered_df[['companyId', 'companyName', 'industryName']].drop_duplicates()

df26 = df_2026.copy()
df25 = df_2025.copy()
for c in PREDICTION_COLS:
    df26[c] = pd.to_numeric(df26[c], errors='coerce').astype(float)
    df25[c] = pd.to_numeric(df25[c], errors='coerce').astype(float)

df26 = df26.merge(name_lookup, on='companyId', how='left')
df25 = df25.merge(name_lookup, on='companyId', how='left')

comp = df25[['companyId', 'companyName', 'industryName'] + PREDICTION_COLS].merge(
    df26[['companyId'] + PREDICTION_COLS], on='companyId', suffixes=['_2025', '_2026']
)
print(f"Data validation: df26={len(df26)}, df25={len(df25)}, comp={len(comp)}")
print(f"Sample values (2026 PredictedMarketingBudget): {df26['PredictedMarketingBudget'].head(3).tolist()}")

COL = 'PredictedMarketingBudget'

def fmt(v):
    if abs(v) >= 1e9: return f"${v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"${v/1e6:.0f}M"
    if abs(v) >= 1e3: return f"${v/1e3:.0f}K"
    return f"${v:.0f}"

# ═══════════════════════════════════════════════════════════════════════
# 1. SCATTER: 2025 vs 2026 (explicit go.Scattergl for reliability)
# ═══════════════════════════════════════════════════════════════════════
x_vals = comp[f'{COL}_2025'].astype(float).values
y_vals = comp[f'{COL}_2026'].astype(float).values
mask = (x_vals > 0) & (y_vals > 0) & np.isfinite(x_vals) & np.isfinite(y_vals)
x_plot, y_plot = x_vals[mask], y_vals[mask]
names_plot = comp.loc[mask, 'companyName'].values
ind_plot = comp.loc[mask, 'industryName'].values

rho_budget = spearmanr(x_plot, y_plot)[0]
max_val = max(x_plot.max(), y_plot.max())
min_val = min(x_plot.min(), y_plot.min())

fig_scatter = go.Figure()
fig_scatter.add_trace(go.Scattergl(
    x=x_plot, y=y_plot, mode='markers',
    marker=dict(size=4, color='#10b981', opacity=0.35),
    text=[f"{n}<br>{i}" for n, i in zip(names_plot, ind_plot)],
    hovertemplate='%{text}<br>2025: $%{x:,.0f}<br>2026: $%{y:,.0f}<extra></extra>',
    name='Companies',
))
fig_scatter.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val], mode='lines',
    line=dict(dash='dash', color='rgba(0,0,0,0.3)', width=1.5), name='y = x',
))
fig_scatter.update_layout(
    title=f'Predicted Marketing Budget: 2025 vs 2026 (n={len(x_plot):,}, Spearman ρ = {rho_budget:.3f})',
    xaxis=dict(title='2025 Predicted Budget ($)', type='log'),
    yaxis=dict(title='2026 Predicted Budget ($)', type='log'),
    template='plotly_white', height=550, showlegend=False,
)
print(f"Scatter: {len(x_plot):,} points plotted, ρ={rho_budget:.3f}")

# ═══════════════════════════════════════════════════════════════════════
# 2. DISTRIBUTION OVERLAY (log₁₀ scale)
# ═══════════════════════════════════════════════════════════════════════
fig_dist = go.Figure()
for year, d, color in [('2025', df25, '#10b981'), ('2026', df26, '#BEF264')]:
    raw = d[COL].astype(float).replace([np.inf, -np.inf, 0], np.nan).dropna()
    vals = np.log10(raw.values)
    print(f"Histogram {year}: {len(vals):,} values, range [{vals.min():.1f}, {vals.max():.1f}]")
    fig_dist.add_trace(go.Histogram(
        x=vals, name=year, opacity=0.55, marker_color=color, nbinsx=80,
    ))
fig_dist.update_layout(
    title='Distribution of Predicted Marketing Budget: 2025 vs 2026 (Log₁₀ Scale)',
    barmode='overlay', template='plotly_white', height=420,
    xaxis_title='log₁₀(Predicted Marketing Budget)', yaxis_title='# Companies',
    legend=dict(x=0.02, y=0.95),
)

# ═══════════════════════════════════════════════════════════════════════
# 3. SPEARMAN RANK STABILITY across all prediction columns
# ═══════════════════════════════════════════════════════════════════════
spearman_data = []
for col in PREDICTION_COLS:
    c25 = comp[f'{col}_2025'].astype(float)
    c26 = comp[f'{col}_2026'].astype(float)
    valid = c25.notna() & c26.notna() & (c25 > 0) & (c26 > 0)
    rho, _ = spearmanr(c25[valid], c26[valid])
    label = col.replace('Predicted', '').replace('Spend', ' Spend').replace('Expense', ' Expense').replace('Budget', ' Budget')
    spearman_data.append({'Metric': label.strip(), 'rho': round(rho, 4)})
spearman_df = pd.DataFrame(spearman_data).sort_values('rho')

fig_spearman = go.Figure(go.Bar(
    x=spearman_df['rho'], y=spearman_df['Metric'], orientation='h',
    text=spearman_df['rho'].apply(lambda x: f'{x:.3f}'), textposition='outside',
    marker_color=spearman_df['rho'].apply(lambda x: f'rgba(39,174,96,{0.4 + 0.6*(x-0.7)/0.3})'),
))
fig_spearman.update_layout(
    title='Year-over-Year Rank Stability (Spearman ρ)', template='plotly_white',
    height=360, xaxis_range=[0.75, 1.0], xaxis_title='Spearman ρ', yaxis_title='',
)

# ═══════════════════════════════════════════════════════════════════════
# 4. INDUSTRY BREAKDOWN (min 10 companies to avoid small-sample noise)
# ═══════════════════════════════════════════════════════════════════════
MIN_INDUSTRY_N = 10
sector_agg = df26.groupby('industryName')[COL].agg(['median', 'count', 'mean', 'sum']).reset_index()
sector_agg.columns = ['Industry', 'Median', 'Count', 'Mean', 'Total']
sector_agg = sector_agg[sector_agg['Count'] >= MIN_INDUSTRY_N]
sector_top = sector_agg.nlargest(15, 'Median').sort_values('Median')

fig_sector = go.Figure(go.Bar(
    x=sector_top['Median'], y=sector_top['Industry'], orientation='h',
    text=sector_top['Median'].apply(fmt), textposition='outside',
    marker_color='#10b981',
    customdata=np.stack([sector_top['Count'], sector_top['Mean'].apply(fmt)], axis=1),
    hovertemplate='<b>%{y}</b><br>Median: %{text}<br>Mean: %{customdata[1]}<br>Companies: %{customdata[0]}<extra></extra>',
))
max_sector_val = sector_top['Median'].max()
fig_sector.update_layout(
    title=dict(text=f'Top 15 Industries by Median Marketing Budget<br><sup>(2026, industries with ≥{MIN_INDUSTRY_N} companies)</sup>'),
    height=560, template='plotly_white', yaxis_title='',
    xaxis_title='Median Predicted Marketing Budget ($)',
    xaxis_range=[0, max_sector_val * 1.5],
    margin=dict(l=260, r=140),
    yaxis=dict(tickfont=dict(size=11)),
)
print(f"Industry chart: {len(sector_top)} industries with ≥{MIN_INDUSTRY_N} companies")

# ═══════════════════════════════════════════════════════════════════════
# 5. CHANNEL ALLOCATION PIE (per-company proportions, then averaged)
# ═══════════════════════════════════════════════════════════════════════
channel_cols = ['PredictedInfluencerSpend', 'PredictedSponsorshipSpend',
                'PredictedAdvertisingSpend', 'PredictedResidualSpend']
channel_labels = ['Influencer', 'Sponsorship', 'Advertising', 'Residual']
channel_colors = ['#10b981', '#F7B500', '#34D399', '#848485']

ch_df = df26[channel_cols].apply(pd.to_numeric, errors='coerce').astype(float)
ch_total = ch_df.sum(axis=1)
ch_props = ch_df.div(ch_total, axis=0)
mean_props = ch_props.mean()
mean_props_pct = (mean_props / mean_props.sum() * 100)

fig_pie = go.Figure(go.Pie(
    labels=channel_labels,
    values=mean_props_pct.values,
    marker_colors=channel_colors,
    textinfo='label+percent', textposition='inside',
    hovertemplate='<b>%{label}</b><br>%{percent}<extra></extra>',
    hole=0.35,
))
fig_pie.update_layout(
    title='Average Marketing Spend Allocation by Channel (2026)',
    template='plotly_white', height=420, showlegend=True,
    legend=dict(orientation='h', y=-0.05),
)
print(f"Channel proportions: {dict(zip(channel_labels, mean_props_pct.round(1).values))}")

# ═══════════════════════════════════════════════════════════════════════
# 6. TOP 30 COMPANIES TABLE
# ═══════════════════════════════════════════════════════════════════════
EXCLUDE_COMPANIES = ['Mycronic AB']
top_pool = comp[~comp['companyName'].isin(EXCLUDE_COMPANIES)]
top_n = top_pool.nlargest(30, f'{COL}_2026').copy()
top_n['v25'] = top_n[f'{COL}_2025'].astype(float)
top_n['v26'] = top_n[f'{COL}_2026'].astype(float)
top_n['yoy'] = ((top_n['v26'] - top_n['v25']) / top_n['v25'] * 100)

yoy_colors = ['#e8f5e9' if v >= 0 else '#fce4ec' for v in top_n['yoy']]
fig_top = go.Figure(data=[go.Table(
    columnwidth=[220, 170, 130, 130, 90],
    header=dict(
        values=['<b>Company</b>', '<b>Industry</b>', '<b>2025 Budget</b>', '<b>2026 Budget</b>', '<b>YoY %</b>'],
        fill_color='#0C1E1B', font=dict(color='white', size=12), align='left', height=32,
    ),
    cells=dict(
        values=[
            top_n['companyName'].values,
            top_n['industryName'].values,
            [fmt(v) for v in top_n['v25']],
            [fmt(v) for v in top_n['v26']],
            [f'{v:+.1f}%' for v in top_n['yoy']],
        ],
        fill_color=[
            [('#f8f9fa' if i % 2 == 0 else 'white') for i in range(len(top_n))],
            [('#f8f9fa' if i % 2 == 0 else 'white') for i in range(len(top_n))],
            [('#f8f9fa' if i % 2 == 0 else 'white') for i in range(len(top_n))],
            [('#f8f9fa' if i % 2 == 0 else 'white') for i in range(len(top_n))],
            yoy_colors,
        ],
        align='left', font=dict(size=11), height=28,
    ),
)])
fig_top.update_layout(title='Top 30 Companies by Predicted Marketing Budget (2026)', height=950, template='plotly_white')

# ═══════════════════════════════════════════════════════════════════════
# 7. YOY CHANGE DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════
yoy_all = ((comp[f'{COL}_2026'].astype(float) - comp[f'{COL}_2025'].astype(float))
           / comp[f'{COL}_2025'].astype(float) * 100)
yoy_clipped = yoy_all.clip(-200, 200).dropna()
fig_yoy = go.Figure(go.Histogram(
    x=yoy_clipped, nbinsx=100, marker_color='#7363f3', opacity=0.75,
))
fig_yoy.add_vline(x=0, line_dash='dash', line_color='red', line_width=1.5)
fig_yoy.add_vline(x=yoy_clipped.median(), line_dash='dot', line_color='green', line_width=1.5)
fig_yoy.add_annotation(
    x=yoy_clipped.median(), y=1.12, yref='paper',
    text=f'<b>Median: {yoy_clipped.median():.1f}%</b>', showarrow=False,
    font=dict(size=13, color='green'),
)
fig_yoy.update_layout(
    title='Distribution of YoY % Change in Predicted Marketing Budget',
    template='plotly_white', height=400,
    xaxis_title='Year-over-Year Change (%)', yaxis_title='# Companies',
)

# ═══════════════════════════════════════════════════════════════════════
# BUILD HTML
# ═══════════════════════════════════════════════════════════════════════
today_str = dt.date.today().strftime('%B %d, %Y')
n_companies = len(df26)
median_budget_26 = float(df26[COL].median())
median_budget_25 = float(df25[COL].median())
median_ratio = median_budget_26 / median_budget_25
mean_rho = np.mean([d['rho'] for d in spearman_data])
channel_str = ', '.join([f"{l}: {v:.1f}%" for l, v in zip(channel_labels, mean_props_pct)])

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>UTA Marketing Budget Model Report</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  *{{margin:0;padding:0;box-sizing:border-box}}
  :root{{--green:#10b981;--lime:#BEF264;--green-light:#d1fae5;--green-dark:#166534;
    --dark:#0C1E1B;--text:rgba(9,10,11,.85);--text2:rgba(9,10,11,.5);
    --grey:#848485;--bg:#ffffff;--card:#fafafa;--border:#e5e5e5}}
  body{{font-family:'Inter',-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    background:var(--bg);color:#090a0b;line-height:1.7;-webkit-font-smoothing:antialiased}}
  .hero{{background:var(--dark);color:#fff;padding:56px 40px 50px;text-align:center;position:relative;overflow:hidden}}
  .hero::before{{content:'';position:absolute;top:-200px;right:-100px;width:500px;height:500px;
    background:radial-gradient(circle,rgba(16,185,129,.12),transparent 70%);border-radius:50%}}
  .hero h1{{font-size:40px;font-weight:800;letter-spacing:-.03em;line-height:1.15;margin-bottom:10px;position:relative;z-index:1}}
  .hero .sub{{font-size:15px;opacity:.55;max-width:600px;margin:0 auto;position:relative;z-index:1}}
  .hero .meta{{font-size:12px;opacity:.35;margin-top:10px;position:relative;z-index:1}}
  .container{{max-width:900px;margin:0 auto;padding:32px 24px}}
  .kpi-row{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px;margin-bottom:32px}}
  .kpi{{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:20px 16px;text-align:center}}
  .kpi .value{{font-size:28px;font-weight:800;letter-spacing:-.02em;color:var(--green)}}
  .kpi .label{{font-size:11px;color:var(--grey);margin-top:4px;text-transform:uppercase;letter-spacing:.5px}}
  .section{{background:#fff;border:1px solid var(--border);border-radius:12px;padding:32px;margin-bottom:20px}}
  .section h2{{font-size:20px;font-weight:700;margin-bottom:8px;color:#090a0b;border-bottom:2px solid var(--green);padding-bottom:6px;display:inline-block}}
  .section p,.section li{{margin:8px 0;color:var(--text);font-size:14px;line-height:1.7}}
  .section ul{{margin-left:20px}}
  .chart{{width:100%;margin-top:12px}}
  .two-col{{display:grid;grid-template-columns:1fr 1fr;gap:20px}}
  @media(max-width:800px){{.two-col{{grid-template-columns:1fr}}}}
  .metric-table{{width:100%;border-collapse:collapse;margin:12px 0;font-size:13px}}
  .metric-table th{{background:var(--dark);color:#fff;padding:10px 12px;text-align:left;font-weight:600}}
  .metric-table td{{padding:8px 12px;border-bottom:1px solid var(--border)}}
  .metric-table tr:nth-child(even){{background:var(--card)}}
  .callout{{background:#f0fdf4;border-left:4px solid var(--green);padding:14px 18px;border-radius:6px;margin:16px 0;font-size:13px;color:var(--text)}}
  .footer{{text-align:center;padding:28px;color:var(--grey);font-size:12px;border-top:1px solid var(--border)}}
</style>
</head>
<body>

<div class="hero">
  <h1>UTA Marketing Budget Model</h1>
  <p class="sub">Predictive marketing spend estimates for {n_companies:,} Brand Ranker companies. ML-generated estimates designed for relative comparisons and ranking.</p>
  <div class="meta">CRED Intelligence Lab &bull; Data Science Team &bull; {today_str}</div>
</div>

<div class="container">

<!-- KPIs -->
<div class="kpi-row">
  <div class="kpi"><div class="value">{n_companies:,}</div><div class="label">Companies Analyzed</div></div>
  <div class="kpi"><div class="value">{fmt(median_budget_26)}</div><div class="label">Median Budget (2026)</div></div>
  <div class="kpi"><div class="value">{median_ratio:.4f}x</div><div class="label">Median YoY Ratio</div></div>
  <div class="kpi"><div class="value">{mean_rho:.3f}</div><div class="label">Avg Spearman ρ</div></div>
</div>

<!-- ────────────────── MODEL METHODOLOGY ────────────────── -->
<div class="section">
  <h2>Model Methodology</h2>
  <p>The <b>Marketing Budget Prediction Model</b> <b>predicts (extrapolates)</b> how much a company likely spends on marketing and how that budget is allocated across advertising, influencer marketing, sponsorship, and residual channels. These are predictive estimates designed for relative comparisons and ranking, not audited actuals. The model is built for companies where actual marketing spend data is unavailable &mdash; which is the vast majority of companies in the CRED database.</p>

  <p><b>How it works:</b></p>
  <ul>
    <li><b>Four LightGBM regression models</b> are trained independently, each predicting a log-transformed target: total marketing expense, influencer spend, advertising spend, and sponsorship spend.</li>
    <li>Each model uses <b>40 predictor features</b> spanning financials (revenue, EBITDA, margins), scale metrics (employees, funding), digital footprint (web traffic, ad impressions, MAU), and categorical identifiers (sector, industry, public/private status).</li>
    <li><b>Hyperparameter tuning</b> is performed via Optuna (Bayesian optimization with TPE sampler and median pruning, 25 trials per target), and models are trained with <b>5-fold cross-validation</b> with LightGBM early stopping.</li>
    <li>All targets are <b>log-transformed</b> (log1p) because marketing spend spans several orders of magnitude &mdash; $100K for a small tech startup vs. $10B for a global CPG brand. Log space ensures the model treats proportional errors equally across company sizes.</li>
    <li>The final <b>top-down allocation</b> step predicts total marketing expense first, then distributes it across channels using model-predicted weights blended with industry-benchmark priors (70% prior / 30% model) to correct for systematic measurement biases across data sources.</li>
  </ul>

  <table class="metric-table">
    <tr><th>Target Model</th><th>Training Rows</th><th>MAE (log)</th><th>R² (log)</th><th>Interpretation</th></tr>
    <tr><td>Marketing Expense</td><td>1,647</td><td>1.316</td><td>0.484</td><td>Predictions typically within ~3.7x of actual</td></tr>
    <tr><td>Influencer Spend</td><td>22,057</td><td>1.991</td><td>0.316</td><td>Hardest target; sparse, noisy ground truth</td></tr>
    <tr><td>Advertising Spend</td><td>83,226</td><td>1.028</td><td>0.731</td><td>Best model; largest training set</td></tr>
    <tr><td>Sponsorship Spend</td><td>7,761</td><td>1.327</td><td>0.266</td><td>Lowest coverage; high variance in reported values</td></tr>
  </table>

  <div class="callout">
    <b>Why log-space metrics?</b> A log-space MAE of 1.0 means predictions are typically off by a factor of e¹ ≈ 2.7x. In dollar terms, the model might predict $5M for a company that actually spends $2M or $14M. This sounds large, but marketing spend is one of the most opaque corporate figures &mdash; even analyst estimates routinely differ by 2&ndash;3x. The advertising model (R²=0.731, MAE=1.028) achieves strong performance, particularly given that no direct marketing budget data is used as input.
  </div>

  <p><b>Data sources:</b> Company financials (income statements), digital ad spend data, influencer pricing data, sponsorship datasets, and company classification data.</p>
</div>

<!-- ────────────────── YOY CONSISTENCY ────────────────── -->
<div class="section">
  <h2>Year-over-Year Consistency</h2>
  <p>A key question for any predictive model is: <i>are the predictions stable across time?</i> Below we compare each company's predicted marketing budget in 2025 vs 2026. Points clustering along the diagonal (y = x) indicate that the model assigns similar budgets year-over-year when company fundamentals haven't changed significantly.</p>
  <div class="chart">{fig_scatter.to_html(full_html=False, include_plotlyjs=False)}</div>
  <div class="callout">
    <b>Spearman ρ = {rho_budget:.3f}</b> &mdash; The rank ordering of companies is highly stable across years. A company ranked in the top 100 in 2025 is overwhelmingly likely to remain in the top 100 in 2026. This is critical for prospecting and prioritization workflows.
  </div>
</div>

<!-- ────────────────── DISTRIBUTIONS ────────────────── -->
<div class="section">
  <h2>Distribution Comparison: 2025 vs 2026</h2>
  <p>The overlaid histograms below show the full distribution of predicted marketing budgets on a log₁₀ scale. Near-complete overlap between 2025 and 2026 confirms there is <b>no systematic inflation or deflation</b> of predictions &mdash; the model hasn't drifted.</p>
  <div class="chart">{fig_dist.to_html(full_html=False, include_plotlyjs=False)}</div>
</div>

<!-- ────────────────── YOY CHANGE DISTRIBUTION ────────────────── -->
<div class="section">
  <h2>Year-over-Year Change Distribution</h2>
  <p>While aggregate distributions are stable, individual companies can show meaningful YoY changes when their underlying financials shift. The histogram below shows the distribution of company-level percentage changes (clipped to ±200% for readability). The median change of <b>{yoy_clipped.median():.1f}%</b> confirms no systematic bias, while the spread reflects legitimate changes in company fundamentals between the two prediction periods.</p>
  <div class="chart">{fig_yoy.to_html(full_html=False, include_plotlyjs=False)}</div>
</div>

<!-- ────────────────── RANK STABILITY ────────────────── -->
<div class="section">
  <h2>Rank Stability Across All Metrics</h2>
  <p>Spearman rank correlations measure whether the relative ordering of companies is preserved year-over-year. All seven prediction columns achieve ρ &gt; 0.80, meaning the model's rankings are robust and trustworthy for comparative analysis and prospecting.</p>
  <div class="chart">{fig_spearman.to_html(full_html=False, include_plotlyjs=False)}</div>
</div>

<!-- ────────────────── INDUSTRY + CHANNEL SIDE BY SIDE ────────────────── -->
<div class="section">
  <h2>Industry Breakdown &amp; Channel Allocation</h2>
  <div class="two-col">
    <div>
      <h3 style="font-size:1rem; color:#090a0b; margin-bottom:6px;">Top Industries by Median Budget</h3>
      <p style="font-size:0.88rem;">Industries with ≥{MIN_INDUSTRY_N} companies, ranked by median predicted marketing budget. Industries with large consumer bases and high advertising intensity naturally dominate.</p>
      <div class="chart">{fig_sector.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>
    <div>
      <h3 style="font-size:1rem; color:#090a0b; margin-bottom:6px;">Average Channel Allocation</h3>
      <p style="font-size:0.88rem;">How the average company's predicted marketing expense is distributed. The allocation reflects a blend of model predictions and industry priors: {channel_str}. "Residual" captures traditional media, events, PR, content marketing, and other non-tracked channels.</p>
      <div class="chart">{fig_pie.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>
  </div>
</div>

<!-- ────────────────── TOP COMPANIES ────────────────── -->
<div class="section">
  <h2>Top 30 Companies by Predicted Marketing Budget (2026)</h2>
  <p>The largest predicted marketing spenders, with side-by-side 2025/2026 comparison. These are predominantly large public companies with substantial revenue and high consumer visibility. YoY changes (green = increase, red = decrease) are driven by shifts in input features — revenue growth/contraction, changes in digital footprint, or updated financial filings.</p>
  <div class="chart">{fig_top.to_html(full_html=False, include_plotlyjs=False)}</div>
</div>

<!-- ────────────────── KEY TAKEAWAYS ────────────────── -->
<div class="section">
  <h2>Key Takeaways</h2>
  <ul>
    <li><b>Model stability is strong:</b> Median YoY ratio of {median_ratio:.4f}x and average Spearman ρ of {mean_rho:.3f} across all metrics confirm that predictions are consistent and reliable for year-over-year comparisons.</li>
    <li><b>Rankings are trustworthy:</b> The rank ordering of companies by predicted marketing spend is highly preserved across years — important for prospecting and competitive benchmarking.</li>
    <li><b>Advertising model is the strongest:</b> With R²=0.731 and the largest training set (83K companies), the advertising spend sub-model drives the most reliable component of the allocation.</li>
    <li><b>Large companies dominate the top:</b> The top 30 predicted spenders are overwhelmingly well-known public companies with revenues in the tens of billions — validating that the model correctly captures the scale relationship between company size and marketing spend.</li>
    <li><b>Company-level deltas are expected:</b> Individual companies can show ±50–100% YoY changes, which is a natural consequence of (a) log-space modeling where small shifts amplify in dollar terms, and (b) legitimate changes in underlying financials between years.</li>
  </ul>
</div>

</div>

<div class="footer">
  CRED Intelligence Lab &bull; Predictive estimates &bull; cred.ai
</div>

</body>
</html>"""

out_path = os.path.join(os.getcwd(), "uta_marketing_budget_report.html")
with open(out_path, "w") as f:
    f.write(html)
print(f"\nReport saved to {out_path}")
webbrowser.open(f"file://{out_path}")

Data validation: df26=8256, df25=8257, comp=8256
Sample values (2026 PredictedMarketingBudget): [82839632.98426476, 84449412.07789458, 102678059.41088255]
Scatter: 8,246 points plotted, ρ=0.818
Histogram 2025: 8,246 values, range [6.0, 12.4]
Histogram 2026: 8,246 values, range [7.8, 11.1]
Industry chart: 15 industries with ≥10 companies
Channel proportions: {'Influencer': np.float64(3.6), 'Sponsorship': np.float64(13.8), 'Advertising': np.float64(18.1), 'Residual': np.float64(64.6)}

Report saved to /Users/sandeepgiri/Documents/cred-data/analysis_notebooks/UTAFieldsModel/uta_marketing_budget_report.html


True